# Multi-Provider API Setup

**Week 2 Day 1 - Learning Lab**

Setting up connections to multiple LLM providers using unified interfaces.

## Intent

Learn to:
- Connect to multiple providers (OpenAI, Anthropic, Gemini, DeepSeek, Groq, Grok, Ollama)
- Use OpenAI-compatible endpoints
- Use native client libraries
- Use abstraction layers (LiteLLM, LangChain, OpenRouter)

## Expected Insights

- Unified interface patterns for model switching
- Tradeoffs between different approaches
- When to use which abstraction layer


In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

# Load API keys
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# Check which keys are available
providers_available = {}
if openai_api_key:
    print(f"✓ OpenAI API Key exists (begins {openai_api_key[:8]}...)")
    providers_available['openai'] = True
else:
    print("✗ OpenAI API Key not set")
    providers_available['openai'] = False

if anthropic_api_key:
    print(f"✓ Anthropic API Key exists (begins {anthropic_api_key[:7]}...)")
    providers_available['anthropic'] = True
else:
    print("✗ Anthropic API Key not set (optional)")
    providers_available['anthropic'] = False

if google_api_key:
    print(f"✓ Google API Key exists (begins {google_api_key[:2]}...)")
    providers_available['gemini'] = True
else:
    print("✗ Google API Key not set (optional)")
    providers_available['gemini'] = False

if deepseek_api_key:
    print(f"✓ DeepSeek API Key exists (begins {deepseek_api_key[:3]}...)")
    providers_available['deepseek'] = True
else:
    print("✗ DeepSeek API Key not set (optional)")
    providers_available['deepseek'] = False

if groq_api_key:
    print(f"✓ Groq API Key exists (begins {groq_api_key[:4]}...)")
    providers_available['groq'] = True
else:
    print("✗ Groq API Key not set (optional)")
    providers_available['groq'] = False

if grok_api_key:
    print(f"✓ Grok API Key exists (begins {grok_api_key[:4]}...)")
    providers_available['grok'] = True
else:
    print("✗ Grok API Key not set (optional)")
    providers_available['grok'] = False

if openrouter_api_key:
    print(f"✓ OpenRouter API Key exists (begins {openrouter_api_key[:3]}...)")
    providers_available['openrouter'] = True
else:
    print("✗ OpenRouter API Key not set (optional)")
    providers_available['openrouter'] = False


## Method 1: OpenAI Client with base_url (Unified Interface)


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai_client = OpenAI(api_key=openai_api_key) if providers_available['openai'] else None

# For Gemini, DeepSeek, Groq, and Grok, we can use the OpenAI python client
# Because they have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

# Create clients for each provider
anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if providers_available['anthropic'] else None
gemini_client = OpenAI(api_key=google_api_key, base_url=gemini_url) if providers_available['gemini'] else None
deepseek_client = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url) if providers_available['deepseek'] else None
groq_client = OpenAI(api_key=groq_api_key, base_url=groq_url) if providers_available['groq'] else None
grok_client = OpenAI(api_key=grok_api_key, base_url=grok_url) if providers_available['grok'] else None
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key) if providers_available['openrouter'] else None

# Check if Ollama is running
try:
    response = requests.get("http://localhost:11434/", timeout=2)
    ollama_available = True
    ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)
    print("✓ Ollama is running locally")
except:
    ollama_available = False
    ollama_client = None
    print("✗ Ollama is not running (optional - run 'ollama serve' to start)")

print("\nAll clients initialized!")


In [ ]:
# Test a simple query across available providers
test_prompt = [
    {"role": "user", "content": "Tell me a joke about LLM engineering in one sentence."}
]

results = {}

# Test OpenAI
if openai_client:
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=test_prompt
        )
        results['OpenAI (GPT-4o-mini)'] = response.choices[0].message.content
        print("✓ OpenAI response received")
    except Exception as e:
        print(f"✗ OpenAI error: {e}")

# Test Gemini via OpenAI client
if gemini_client:
    try:
        response = gemini_client.chat.completions.create(
            model="gemini-2.5-flash-lite",
            messages=test_prompt
        )
        results['Gemini (via OpenAI client)'] = response.choices[0].message.content
        print("✓ Gemini response received")
    except Exception as e:
        print(f"✗ Gemini error: {e}")

# Test DeepSeek
if deepseek_client:
    try:
        response = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            messages=test_prompt
        )
        results['DeepSeek'] = response.choices[0].message.content
        print("✓ DeepSeek response received")
    except Exception as e:
        print(f"✗ DeepSeek error: {e}")

# Test Ollama (if available)
if ollama_client and ollama_available:
    try:
        response = ollama_client.chat.completions.create(
            model="llama3.2",
            messages=test_prompt
        )
        results['Ollama (Llama 3.2)'] = response.choices[0].message.content
        print("✓ Ollama response received")
    except Exception as e:
        print(f"✗ Ollama error: {e}")

# Display results
print("\n" + "="*60)
print("RESULTS:")
print("="*60)
for provider, response in results.items():
    print(f"\n{provider}:")
    display(Markdown(response))
